# ML Development Notebook for DysCalc

## Configs

### Imports

In [25]:
import itertools
import numpy as np
import pandas as pd
from typing import Dict, List
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score


### ML Model Configs

## Functions and Classes

### C4.5 Decision Tree Class Implementation

In [26]:
from C45DecisionTree import (
    C45DecisionTree
)

FUNA_DB_DOMAIN_MAPPING: Dict[str, str] = {
    # --- Number Processing raw tasks ---
    "NC": "Number Comparison",
    "DM": "Digit-Dot Matching",
    # --- Number Processing derived features ---
    "NP": "Overall Processing Efficiency",
    "SN": "Symbolic vs. Non-Symbolic Processing Difference",
    # --- Arithmetic Fluency raw tasks ---
    "NS": "Number Series",
    "ADD": "Single-Digit Addition",
    "SUB": "Single-Digit Subtraction",
    "CA":  "Multi-Digit Addition and Subtraction",
    # --- Arithmetic Fluency derived features ---
    "AF": "Overall Arithmetic Fluency",
    "BC": "Basic vs. Complex Arithmetic Contrast",
    "AS": "Addition vs. Subtraction Asymmetry",
    "PF": "Processing-Fluency Integration",
}
 
# Raw FUNA-DB task features used for tree training (Section 3.1.3 / 3.2.1.1)
FUNA_DB_RAW_FEATURES: List[str] = ["NC", "DM", "NS", "ADD", "SUB", "CA"]
 
# Full 12-feature vector used for diagnostics (Section 3.2.1.5)
FUNA_DB_DIAGNOSTIC_FEATURES: List[str] = [
    "NC", "DM", "NS", "ADD", "SUB", "CA",   # raw tasks
    "NP", "SN", "AF", "BC", "AS", "PF",     # derived features
]

# Convenience helper — splits a DataFrame into features and label
def split_xy(df: pd.DataFrame, label_col: str = 'Label'):
    """Return (X, y) with X restricted to the full diagnostic feature vector Xi."""
    X = df[FUNA_DB_DIAGNOSTIC_FEATURES]
    y = df[label_col]
    return X, y


## Training

### Data Loading & Splitting
Load the complete dataset vector and split it into Train (70%), Validation (15%), and Test (15%).

In [27]:
print('Loading 70/15/15 Datasets...')

# ── Augmented (GAN-balanced) training set  (Sec 3.2.3.1 — TSTR evaluation)
# ── Purely real validation and test sets
train_df = pd.read_csv('GAN/dataset/FUNADB_balanced_TRAIN_1.csv')
val_df   = pd.read_csv('GAN/dataset/FUNADB_real_VAL_1.csv')
test_df  = pd.read_csv('GAN/dataset/FUNADB_real_TEST_1.csv')

print(f'Train shape (Balanced 70%):          {train_df.shape}')
print(f'Validation shape (Purely Real 15%):  {val_df.shape}')
print(f'Test shape (Purely Real 15%):        {test_df.shape}')

# split_xy slices exactly the 12 columns in FUNA_DB_DIAGNOSTIC_FEATURES
# Training uses the GAN-augmented set; val/test use real data only (TSTR)
X_train, y_train = split_xy(train_df, 'Label')
X_val,   y_val   = split_xy(val_df,   'Label')
X_test,  y_test  = split_xy(test_df,  'Label')

print(f'\nFeatures used for training  (raw,  Sec 3.1.3):      {FUNA_DB_RAW_FEATURES}')
print(f'Features used for diagnosis (full, Sec 3.2.1.5):   {FUNA_DB_DIAGNOSTIC_FEATURES}')
print(f'Domain mapping              (Sec 3.1.2):            {FUNA_DB_DOMAIN_MAPPING}')


Loading 70/15/15 Datasets...
Train shape (Balanced 70%):          (308, 13)
Validation shape (Purely Real 15%):  (54, 13)
Test shape (Purely Real 15%):        (54, 13)

Features used for training  (raw,  Sec 3.1.3):      ['NC', 'DM', 'NS', 'ADD', 'SUB', 'CA']
Features used for diagnosis (full, Sec 3.2.1.5):   ['NC', 'DM', 'NS', 'ADD', 'SUB', 'CA', 'NP', 'SN', 'AF', 'BC', 'AS', 'PF']
Domain mapping              (Sec 3.1.2):            {'NC': 'Number Comparison', 'DM': 'Digit-Dot Matching', 'NP': 'Overall Processing Efficiency', 'SN': 'Symbolic vs. Non-Symbolic Processing Difference', 'NS': 'Number Series', 'ADD': 'Single-Digit Addition', 'SUB': 'Single-Digit Subtraction', 'CA': 'Multi-Digit Addition and Subtraction', 'AF': 'Overall Arithmetic Fluency', 'BC': 'Basic vs. Complex Arithmetic Contrast', 'AS': 'Addition vs. Subtraction Asymmetry', 'PF': 'Processing-Fluency Integration'}


### Cross-Validation & Hyperparameter Tuning
Perform Grid Search with 5-fold Stratified Cross-Validation on the training set to optimize `conf_fact`, `min_samples_leaf`, and `max_depth`.

In [28]:
CV_EVAL_THRESHOLD = 0.375  # Default threshold used during cross-validation only

# Hyperparameter grid (Sec 3.2.3.2)
# conf_fact ∈ [0.10, 0.50], min_samples_leaf ∈ [10, 50], max_depth ∈ [5, 15]
param_grid = {
    'conf_fact':        [0.05, 0.10, 0.15, 0.25, 0.35, 0.45],
    'min_samples_leaf': [1, 2, 3, 5, 8],
    'max_depth':        [None, 7, 10, 12, 15],
}

# FUNA_DB_DOMAIN_MAPPING is imported from C45DecisionTree — covers all 12 features
# and maps them to 'Number Processing' or 'Arithmetic Fluency' (Sec 3.1.2, Eq 3.35)
# No need to redefine it here.

keys, values = zip(*param_grid.items())
hyperparams_combos = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f'Total hyperparameter combinations: {len(hyperparams_combos)}')

all_results = []
best_f1 = -1
best_params = None
best_cv_metrics = None

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Starting Grid Search CV...')
for idx, params in enumerate(hyperparams_combos, 1):
    cv_f1, cv_recall, cv_precision, cv_accuracy = [], [], [], []

    for train_index, val_index in skf.split(X_train, y_train):
        X_trn_fold = X_train.iloc[train_index]
        X_val_fold = X_train.iloc[val_index]
        y_trn_fold = y_train.iloc[train_index]
        y_val_fold = y_train.iloc[val_index]

        # FUNA_DB_DOMAIN_MAPPING is the default — no need to pass explicitly,
        # but shown here for transparency.
        tree = C45DecisionTree(**params, feature_domain_mapping=FUNA_DB_DOMAIN_MAPPING)

        # fit() trains on raw 6 task features; stores stats over all 12 for diagnostics
        tree.fit(X_trn_fold, y_trn_fold, raw_features=FUNA_DB_RAW_FEATURES)

        diagnostics = tree.predict_with_diagnostics(X_val_fold)

        # predicted_class is stored as str (e.g. '0' / '1') — cast before comparison
        probs = [d.confidence if int(d.predicted_class) == 1 else 1 - d.confidence
                 for d in diagnostics]
        preds = [1 if p >= CV_EVAL_THRESHOLD else 0 for p in probs]

        cv_f1.append(f1_score(y_val_fold, preds))
        cv_recall.append(recall_score(y_val_fold, preds))
        cv_precision.append(precision_score(y_val_fold, preds, zero_division=0))
        cv_accuracy.append(accuracy_score(y_val_fold, preds))

    mean_f1     = np.mean(cv_f1)
    mean_recall = np.mean(cv_recall)
    std_f1      = np.std(cv_f1)
    std_recall  = np.std(cv_recall)

    all_results.append({
        'params':           params,
        'mean_recall':      mean_recall,
        'mean_f1':          mean_f1,
        'mean_precision':   np.mean(cv_precision),
        'mean_accuracy':    np.mean(cv_accuracy),
    })

    if idx % 50 == 0:
        print(f'  Evaluated {idx}/{len(hyperparams_combos)} combinations...')

    # Primary criterion: recall ≥ 0.85 (Sec 3.2.3.2), secondary: maximise F1
    if mean_recall >= 0.85 and mean_f1 > best_f1:
        best_f1     = mean_f1
        best_params = params
        best_cv_metrics = {
            'mean_f1':          mean_f1,          'std_f1':          std_f1,
            'mean_recall':      mean_recall,       'std_recall':      std_recall,
            'mean_precision':   np.mean(cv_precision), 'std_precision': np.std(cv_precision),
            'mean_accuracy':    np.mean(cv_accuracy),  'std_accuracy':  np.std(cv_accuracy),
        }

# ── Fallback handling ──────────────────────────────────────────────────────
valid_results = [r for r in all_results if r['mean_recall'] >= 0.85]
all_results_sorted = (
    sorted(valid_results, key=lambda x: x['mean_f1'], reverse=True)
    if valid_results
    else sorted(all_results, key=lambda x: x['mean_recall'], reverse=True)
)
max_recall_achieved = all_results_sorted[0]['mean_recall']

print(f'\n=== Grid Search Summary ===')
print(f'Maximum recall achieved: {max_recall_achieved:.4f}')
print(f'\nTop 5 configurations by F1 (recall ≥ 0.85 preferred):')
for i, result in enumerate(all_results_sorted[:5], 1):
    print(f'  {i}. Recall={result["mean_recall"]:.4f}, F1={result["mean_f1"]:.4f}, '
          f'Params={result["params"]}')

if best_params is None:
    print('\nWarning: No combination achieved recall >= 0.85.')
    if max_recall_achieved >= 0.85:
        print(f'Using highest-recall config ({max_recall_achieved:.4f})...')
        best_result = all_results_sorted[0]
        best_params = best_result['params']
        cv_f1, cv_recall, cv_precision, cv_accuracy = [], [], [], []
        for train_index, val_index in skf.split(X_train, y_train):
            X_trn_fold = X_train.iloc[train_index]
            X_val_fold = X_train.iloc[val_index]
            y_trn_fold = y_train.iloc[train_index]
            y_val_fold = y_train.iloc[val_index]
            tree = C45DecisionTree(**best_params, feature_domain_mapping=FUNA_DB_DOMAIN_MAPPING)
            tree.fit(X_trn_fold, y_trn_fold, raw_features=FUNA_DB_RAW_FEATURES)
            diagnostics = tree.predict_with_diagnostics(X_val_fold)
            probs = [d.confidence if int(d.predicted_class) == 1 else 1 - d.confidence
                     for d in diagnostics]
            preds = [1 if p >= CV_EVAL_THRESHOLD else 0 for p in probs]
            cv_f1.append(f1_score(y_val_fold, preds))
            cv_recall.append(recall_score(y_val_fold, preds))
            cv_precision.append(precision_score(y_val_fold, preds, zero_division=0))
            cv_accuracy.append(accuracy_score(y_val_fold, preds))
        best_cv_metrics = {
            'mean_f1': np.mean(cv_f1),   'std_f1': np.std(cv_f1),
            'mean_recall': np.mean(cv_recall), 'std_recall': np.std(cv_recall),
            'mean_precision': np.mean(cv_precision), 'std_precision': np.std(cv_precision),
            'mean_accuracy': np.mean(cv_accuracy), 'std_accuracy': np.std(cv_accuracy),
        }
    else:
        print(f'Max recall ({max_recall_achieved:.4f}) below 0.80 — consider feature '
              'engineering or class balancing.')
        best_params = all_results_sorted[0]['params']

print(f'\n=== Best Hyperparameters ===')
print(f'  Confidence Factor (conf_fact):   {best_params["conf_fact"]}')
print(f'  Min Samples Leaf:                {best_params["min_samples_leaf"]}')
print(f'  Max Depth:                       {best_params["max_depth"]}')

print(f'\n⚠ Note: CV metrics above are evaluated at {CV_EVAL_THRESHOLD} threshold.')
print(f'   The final threshold will be tuned on the real validation set.')

if best_cv_metrics:
    print(f'\n=== Cross-Validation Performance (5-Fold, Sec 3.2.3.3) ===')
    print(f'  Recall:    {best_cv_metrics["mean_recall"]:.4f} ± {best_cv_metrics["std_recall"]:.4f}')
    print(f'  Precision: {best_cv_metrics["mean_precision"]:.4f} ± {best_cv_metrics["std_precision"]:.4f}')
    print(f'  F1-Score:  {best_cv_metrics["mean_f1"]:.4f} ± {best_cv_metrics["std_f1"]:.4f}')
    print(f'  Accuracy:  {best_cv_metrics["mean_accuracy"]:.4f} ± {best_cv_metrics["std_accuracy"]:.4f}')


Total hyperparameter combinations: 150
Starting Grid Search CV...
  Evaluated 50/150 combinations...
  Evaluated 100/150 combinations...
  Evaluated 150/150 combinations...

=== Grid Search Summary ===
Maximum recall achieved: 0.8445

Top 5 configurations by F1 (recall ≥ 0.85 preferred):
  1. Recall=0.8445, F1=0.6227, Params={'conf_fact': 0.05, 'min_samples_leaf': 1, 'max_depth': 10}
  2. Recall=0.8381, F1=0.6212, Params={'conf_fact': 0.1, 'min_samples_leaf': 1, 'max_depth': 10}
  3. Recall=0.8381, F1=0.6212, Params={'conf_fact': 0.15, 'min_samples_leaf': 1, 'max_depth': 10}
  4. Recall=0.8381, F1=0.6212, Params={'conf_fact': 0.25, 'min_samples_leaf': 1, 'max_depth': 10}
  5. Recall=0.8381, F1=0.6212, Params={'conf_fact': 0.35, 'min_samples_leaf': 1, 'max_depth': 10}

Max recall (0.8445) below 0.80 — consider feature engineering or class balancing.

=== Best Hyperparameters ===
  Confidence Factor (conf_fact):   0.05
  Min Samples Leaf:                1
  Max Depth:                    

### Final Model Training & Evaluation
Train the optimal model on the entire training set and evaluate on Validation and Test sets.

In [29]:
def evaluate_model(y_true, y_pred, y_probs=None):
    """Compute standard classification metrics."""
    metrics = {
        'Recall':    recall_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'F1-Score':  f1_score(y_true, y_pred),
        'Accuracy':  accuracy_score(y_true, y_pred),
    }
    if y_probs is not None:
        metrics['AUC'] = roc_auc_score(y_true, y_probs)
    return metrics

# Train final model on the full (augmented) training set using the best hyperparams
final_tree = C45DecisionTree(**best_params, feature_domain_mapping=FUNA_DB_DOMAIN_MAPPING)
print(f"{best_params=}")
# fit() trains on FUNA_DB_RAW_FEATURES (6 raw task scores, Sec 3.1.3)
# and computes feature_stats over all 12 columns for diagnostic scoring (Sec 3.2.4)
final_tree.fit(X_train, y_train, raw_features=FUNA_DB_RAW_FEATURES)

print(f"\nFinal model trained on {len(X_train)} augmented samples.")


best_params={'conf_fact': 0.05, 'min_samples_leaf': 1, 'max_depth': 10}

Final model trained on 308 augmented samples.


### Threshold Tuning on Real Validation Set
Find the optimal decision threshold that maximizes F1 while achieving recall ≥ 0.85, evaluated on the real validation set only.


In [36]:
import numpy as np
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score

# =========================
# CONFIGURATION
# =========================
MIN_RECALL = 0.85
MIN_PRECISION = 0.50  # adjust based on acceptable false positive rate

# =========================
# GET VALIDATION PROBABILITIES
# =========================
val_diagnostics = final_tree.predict_with_diagnostics(X_val)

# IMPORTANT FIX: use proper probability directly
val_probs = np.array([d.confidence for d in val_diagnostics])

# =========================
# THRESHOLD SEARCH SPACE
# =========================
threshold_candidates = np.linspace(0.2, 0.8, 31)

threshold_results = []

# =========================
# EVALUATE THRESHOLDS
# =========================
for threshold in threshold_candidates:
    preds = (val_probs >= threshold).astype(int)

    recall = recall_score(y_val, preds, zero_division=0)
    precision = precision_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds, zero_division=0)
    accuracy = accuracy_score(y_val, preds)

    threshold_results.append({
        "threshold": threshold,
        "recall": recall,
        "precision": precision,
        "f1": f1,
        "accuracy": accuracy,
        "meets_constraints": (
            recall >= MIN_RECALL and precision >= MIN_PRECISION
        )
    })

# =========================
# FILTER VALID THRESHOLDS
# =========================
valid_thresholds = [
    r for r in threshold_results if r["meets_constraints"]
]

# =========================
# SELECT BEST THRESHOLD
# =========================
if valid_thresholds:
    best_threshold_config = max(valid_thresholds, key=lambda x: x["f1"])
    optimal_threshold = best_threshold_config["threshold"]
    print("✓ Found threshold that meets recall + precision constraints")

else:
    best_threshold_config = max(threshold_results, key=lambda x: x["recall"])
    optimal_threshold = best_threshold_config["threshold"]

    print("⚠ Warning: No threshold meets both constraints")
    print("  Falling back to highest-recall threshold")

# =========================
# OUTPUT BEST RESULT
# =========================
print("\n=== Optimal Threshold (Tuned on Real Validation) ===")
print(f"  Threshold: {optimal_threshold:.3f}")
print(f"  Recall:    {best_threshold_config['recall']:.4f}")
print(f"  Precision: {best_threshold_config['precision']:.4f}")
print(f"  F1-Score:  {best_threshold_config['f1']:.4f}")
print(f"  Accuracy:  {best_threshold_config['accuracy']:.4f}")

# =========================
# TOP 5 THRESHOLDS
# =========================
if valid_thresholds:
    sorted_results = sorted(valid_thresholds, key=lambda x: x["f1"], reverse=True)
    print("\n=== Top 5 Thresholds (Meeting Constraints) ===")
else:
    sorted_results = sorted(threshold_results, key=lambda x: x["f1"], reverse=True)
    print("\n=== Top 5 Thresholds (No Valid Thresholds Found — Showing All) ===")

for i, config in enumerate(sorted_results[:5], 1):
    print(
        f"  {i}. Threshold={config['threshold']:.3f}, "
        f"Recall={config['recall']:.4f}, "
        f"Precision={config['precision']:.4f}, "
        f"F1={config['f1']:.4f}"
    )

⚠ Warning: No threshold meets both constraints
  Falling back to highest-recall threshold

=== Optimal Threshold (Tuned on Real Validation) ===
  Threshold: 0.200
  Recall:    1.0000
  Precision: 0.3889
  F1-Score:  0.5600
  Accuracy:  0.3889

=== Top 5 Thresholds (No Valid Thresholds Found — Showing All) ===
  1. Threshold=0.200, Recall=1.0000, Precision=0.3889, F1=0.5600
  2. Threshold=0.220, Recall=1.0000, Precision=0.3889, F1=0.5600
  3. Threshold=0.240, Recall=1.0000, Precision=0.3889, F1=0.5600
  4. Threshold=0.260, Recall=1.0000, Precision=0.3889, F1=0.5600
  5. Threshold=0.280, Recall=1.0000, Precision=0.3889, F1=0.5600


### Final Test Evaluation with Tuned Threshold
Evaluate the final model on the real test set using the optimal threshold discovered on validation.


In [31]:
# Evaluate on REAL test set with the optimal threshold
print(f"═" * 60)
print(f"Evaluating on Real Test Set (unseen) with Optimal Threshold")
print(f"═" * 60)

def evaluate_split(tree, X, y_true, label, threshold):
    """Run predict_with_diagnostics and apply threshold; print metrics."""
    diagnostics = tree.predict_with_diagnostics(X)
    probs = [d.confidence if int(d.predicted_class) == 1 else 1 - d.confidence
             for d in diagnostics]
    preds = [1 if p >= threshold else 0 for p in probs]
    metrics = evaluate_model(y_true, preds, probs)
    print(f'\n--- {label} Performance (Threshold: {threshold:.3f}) ---')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')
    return diagnostics, probs, preds

# Test evaluation (unseen data, optimal threshold from validation)
test_diagnostics, test_probs, test_preds = evaluate_split(
    final_tree, X_test, y_test, 'Test Set (REAL, UNSEEN)', optimal_threshold
)

print(f"\n✓ Model locked to threshold: {optimal_threshold:.3f}")
print(f"  This threshold was selected to maximize F1 while meeting recall ≥ 0.85 on validation.")


════════════════════════════════════════════════════════════
Evaluating on Real Test Set (unseen) with Optimal Threshold
════════════════════════════════════════════════════════════

--- Test Set (REAL, UNSEEN) Performance (Threshold: 0.200) ---
  Recall: 0.9524
  Precision: 0.3846
  F1-Score: 0.5479
  Accuracy: 0.3889
  AUC: 0.6674

✓ Model locked to threshold: 0.200
  This threshold was selected to maximize F1 while meeting recall ≥ 0.85 on validation.


### Interpretability & Diagnostic Outputs
Extract Global Feature Importance and generate a sample explanation for an 'At-Risk' prediction.

In [32]:
print('\n--- Global Feature Importance (Eq 3.37) ---')
importance = final_tree.get_feature_importance()
sorted_importance = sorted(importance.items(), key=lambda item: item[1], reverse=True)
for feature, val in sorted_importance:
    print(f'  {feature}: {val:.4f}')

print('\n--- Sample Diagnostic Output (At-Risk case) ---')
# predicted_class is stored as str ('0' or '1') — compare with string '1'
sample_cnt = 0
for idx, diag in enumerate(test_diagnostics):
    if diag.predicted_class == '1':
        print(f'\n  Test Case #{idx}:')
        print(f'    Predicted Class  : At-Risk (1)')
        print(f'    Confidence       : {diag.confidence:.4f}')
        print(f'    Decision Path    : {diag.decision_path_readable}')
        print(f'    Domain Severity  : {diag.domain_severity_scores}')  # Eq 3.35 (wn = IG)
        print(f'    Task Importance  : {diag.task_importance_scores}')  # Eq 3.36 (wn = GR)
        sample_cnt += 1
        
    if sample_cnt > 5:    
        break  # show first 5 At-Risk case only



--- Global Feature Importance (Eq 3.37) ---
  ADD: 0.3161
  NS: 0.1881
  SUB: 0.1694
  DM: 0.1425
  NC: 0.1010
  CA: 0.0830

--- Sample Diagnostic Output (At-Risk case) ---

  Test Case #7:
    Predicted Class  : At-Risk (1)
    Confidence       : 0.8333
    Decision Path    : ADD > 21.5000 AND DM > 3717.7171
    Domain Severity  : {'Single-Digit Subtraction': np.float64(0.0), 'Digit-Dot Matching': np.float64(0.0060574974712607355), 'Number Series': np.float64(0.0), 'Basic vs. Complex Arithmetic Contrast': np.float64(0.0841033135596427), 'Addition vs. Subtraction Asymmetry': np.float64(0.03986855383445434), 'Processing-Fluency Integration': np.float64(0.16426385085060932), 'Overall Processing Efficiency': np.float64(0.29171882060395915), 'Number Comparison': np.float64(0.0), 'Single-Digit Addition': np.float64(0.016100094386179563), 'Multi-Digit Addition and Subtraction': np.float64(0.0), 'Overall Arithmetic Fluency': np.float64(0.12585911960930538), 'Symbolic vs. Non-Symbolic Process

In [33]:
print("\\n--- Serializing Final Model ---")

final_tree.save_model(
    filepath='dyscalc_final_model_1.pkl', 
    optimal_threshold=optimal_threshold  # Uses the tuned threshold from validation
)


\n--- Serializing Final Model ---


In [34]:
from graphviz import Digraph

def generate_tree_image(root_node, filename="decision_tree"):
    """
    Standalone function to visualize a decision tree from any starting node.
    
    :param root_node: The starting node (e.g., final_tree.tree or a specific subtree node)
    :param filename: Output filename
    """
    if root_node is None:
        print("Error: The provided node is None.")
        return

    dot = Digraph(comment='Decision Tree', format='svg')
    dot.attr(rankdir='TB', size='8,5') # Top to Bottom layout
    
    visited = set()

    def add_nodes_edges(node, parent_id=None, edge_label=""):
        if node is None or id(node) in visited:
            return
        
        visited.add(id(node))
        node_id = str(id(node))
        
        # --- Node Styling and Labeling ---
        if node.type == "leaf":
            # Extract distribution if available
            dist = getattr(node, 'distribution', {})
            dist_str = ", ".join([f"{k}:{v}" for k, v in dist.items()])
            
            label = f"LEAF\nClass: {node.label}\nSamples: {node.samples}\n[{dist_str}]"
            dot.node(node_id, label, shape="ellipse", style="filled", color="#BDECB6", fontname="Arial")
        else:
            # Formatting floats for readability
            threshold = f"{node.threshold:.2f}" if hasattr(node, 'threshold') else "N/A"
            gr = f"{node.gain_ratio:.4f}" if hasattr(node, 'gain_ratio') else "N/A"
            
            label = f"{node.feature} <= {threshold}\nGR: {gr}\nSamples: {node.samples}"
            dot.node(node_id, label, shape="box", style="rounded,filled", color="#ADD8E6", fontname="Arial")

        # --- Edge Creation ---
        if parent_id:
            dot.edge(parent_id, node_id, label=edge_label, fontname="Arial", fontsize="10")

        # --- Recursion ---
        if node.type != "leaf":
            if hasattr(node, 'left'):
                add_nodes_edges(node.left, node_id, "True")
            if hasattr(node, 'right'):
                add_nodes_edges(node.right, node_id, "False")

    # Start the process
    add_nodes_edges(root_node)
    
    try:
        output_path = dot.render(filename, view=True, cleanup=True)
        print(f"Successfully saved tree image to: {output_path}")
    except Exception as e:
        print(f"Failed to render Graphviz: {e}")
        print("Ensure Graphviz (the software) is installed and in your system PATH.")

# --- How to call it ---
# To visualize the whole tree:
generate_tree_image(final_tree.tree, "my_full_tree")

# To visualize just a specific branch:
# generate_tree_image(final_tree.tree.left, "left_branch_only")

Successfully saved tree image to: my_full_tree.svg
